# Deployment — pydantic, joblib, FastAPI cheatsheet


Three libraries cover most of "taking a notebook model to production":

- **joblib** — persists trained models to disk, smaller and faster than pickle.
- **pydantic** — validates input data at the API boundary; catches type errors and missing fields *before* they corrupt model predictions.
- **FastAPI** — wraps your model in an HTTP endpoint with auto-generated OpenAPI docs. The patterns here apply equally to Flask, Lambda, etc.

The single most important habit: **bundle the model artifact with feature names**. Every silent inference bug — feature ordering, missing fields, type mismatches — starts with the artifact not knowing what it expects.


---
## Setup

Run this once.


### Setup — run me first


In [1]:
import os
import tempfile
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from pydantic import BaseModel, Field, ValidationError, create_model
from typing import List

rng = np.random.default_rng(0)
X, y = make_classification(n_samples=300, n_features=4, random_state=0)
FEATURE_NAMES = [f'f{i}' for i in range(4)]
X = pd.DataFrame(X, columns=FEATURE_NAMES)
model = LogisticRegression(max_iter=500).fit(X, y)
ARTIFACT_PATH = os.path.join(tempfile.gettempdir(), 'demo_artifact.joblib')

---
## 1. Persisting a model — joblib

joblib is faster and smaller than pickle for numpy-heavy objects (which most models are). Always bundle the model with metadata it depends on.


### How do I save and load a model?

`joblib.dump(obj, path)` / `joblib.load(path)`. Bundle as a dict with metadata, never just the model alone.


In [2]:
artifact = {
    'model': model,
    'feature_names': FEATURE_NAMES,
    'trained_through': '2024-01-31T23:59:59Z',
    'threshold': 0.5,
    'sklearn_version': '1.8.0',
}
joblib.dump(artifact, ARTIFACT_PATH)
loaded = joblib.load(ARTIFACT_PATH)
print(loaded.keys())

dict_keys(['model', 'feature_names', 'trained_through', 'threshold', 'sklearn_version'])


*Why dict bundle*: at inference time you NEED the feature names (for ordering), the threshold (if there is one), and the training cutoff (for staleness checks). All in one artifact = single source of truth. *Common mistake*: saving just the model and hardcoding feature names elsewhere — they drift apart on the next retraining.


### How do I write a stateless predict_one function?

Load artifact, reorder columns by `feature_names`, predict. The reorder is the single critical line.


In [3]:
def predict_one(row: dict, artifact_path: str = ARTIFACT_PATH) -> dict:
    art = joblib.load(artifact_path)
    df = pd.DataFrame([row])[art['feature_names']]    # ← critical: reorder
    prob = float(art['model'].predict_proba(df)[0, 1])
    return {
        'prob_up': prob,
        'label': int(prob >= art['threshold']),
        'trained_through': art['trained_through'],
    }

print(predict_one(X.iloc[0].to_dict()))

{'prob_up': 0.966132958944042, 'label': 1, 'trained_through': '2024-01-31T23:59:59Z'}


*Why the reorder*: dict iteration order may not match training column order. Without the reorder, the model uses the wrong values for each tree split / coefficient. *Common mistake*: omitting the `[art['feature_names']]` slice — silent wrong predictions in production.


---
## 2. Pydantic — input validation

Pydantic v2 enforces types and presence at the API boundary. It rejects malformed input *before* it reaches your model.


### How do I declare a typed input schema?

Subclass `BaseModel` with type-annotated fields.


In [4]:
class PredictRequest(BaseModel):
    f0: float
    f1: float
    f2: float
    f3: float

# Valid input.
req = PredictRequest(f0=1.0, f1=2.0, f2=3.0, f3=4.0)
print(req.model_dump())

# Invalid (string instead of float).
try:
    PredictRequest(f0='not a number', f1=2.0, f2=3.0, f3=4.0)
except ValidationError as e:
    print(f'rejected: {str(e).splitlines()[1]}')

{'f0': 1.0, 'f1': 2.0, 'f2': 3.0, 'f3': 4.0}
rejected: f0


*Why explicit fields*: clearest schema for documentation and IDE autocomplete. *When to use create_model instead*: when the schema is generated from a runtime list (next entry).


### How do I generate a Pydantic schema from a feature list?

`create_model('Name', **{f: (float, ...) for f in feature_names})`.


In [5]:
PredictRequestDynamic = create_model(
    'PredictRequest',
    **{f: (float, ...) for f in FEATURE_NAMES},
)

req = PredictRequestDynamic(**{f: 1.0 for f in FEATURE_NAMES})
print(req.model_dump())

{'f0': 1.0, 'f1': 1.0, 'f2': 1.0, 'f3': 1.0}


*When to use*: feature list comes from the artifact at load time; you don't want to maintain a separate hardcoded schema. *Why `(float, ...)`*: tuple form is `(type, default)`; `...` (`Ellipsis`) means "required, no default".


### How do I add range constraints on a field?

`Field(ge=, le=, gt=, lt=)`. The schema rejects out-of-range inputs.


In [6]:
class BoundedRequest(BaseModel):
    score: float = Field(ge=0.0, le=1.0)
    horizon: int = Field(ge=1, le=168)

# Valid.
print(BoundedRequest(score=0.5, horizon=24).model_dump())

# Invalid (score out of range).
try:
    BoundedRequest(score=2.0, horizon=24)
except ValidationError as e:
    print(f'rejected: {str(e).splitlines()[1]}')

{'score': 0.5, 'horizon': 24}
rejected: score


*When to use*: any field with a known sensible range (probabilities ∈ [0,1], horizons in some max, RSI ∈ [0, 100]). *Common mistake*: relying on "users won't send bad data" — they will, and pydantic catching it at the boundary is much cleaner than the model producing garbage.


---
## 3. FastAPI — wrapping the model

FastAPI uses pydantic models to define request/response shapes and auto-generates OpenAPI docs. The patterns shown work equally well in any web framework.


### How would I structure a /predict endpoint?

Define request and response Pydantic models, decorate the handler with `@app.post`.


In [7]:
# This is illustrative — running FastAPI in a notebook needs uvicorn etc.
# In a script you'd do: from fastapi import FastAPI; app = FastAPI()

handler_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()

class PredictResp(BaseModel):
    prob_up: float
    label: int

@app.post("/predict", response_model=PredictResp)
def predict(req: PredictRequest):
    try:
        result = predict_one(req.model_dump())
        return PredictResp(prob_up=result["prob_up"], label=result["label"])
    except KeyError as e:
        raise HTTPException(400, detail=f"missing feature: {e}")
'''
print(handler_code)


from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()

class PredictResp(BaseModel):
    prob_up: float
    label: int

@app.post("/predict", response_model=PredictResp)
def predict(req: PredictRequest):
    try:
        result = predict_one(req.model_dump())
        return PredictResp(prob_up=result["prob_up"], label=result["label"])
    except KeyError as e:
        raise HTTPException(400, detail=f"missing feature: {e}")



*Why pydantic models on input AND output*: input validation prevents bad requests; output schema gives clients a contract and powers OpenAPI docs. *Common mistake*: returning a raw dict — works, but loses the response schema and the auto-validation.


### How would I structure a /health endpoint?

Return artifact metadata (training cutoff, feature count, model version). Don't expose internals.


In [8]:
def health() -> dict:
    art = joblib.load(ARTIFACT_PATH)
    return {
        'status': 'ok',
        'trained_through': art['trained_through'],
        'n_features': len(art['feature_names']),
        'sklearn_version': art.get('sklearn_version', 'unknown'),
    }

print(health())

{'status': 'ok', 'trained_through': '2024-01-31T23:59:59Z', 'n_features': 4, 'sklearn_version': '1.8.0'}


*What goes in /health*: enough for monitoring + load balancers + clients to assess staleness. *What does NOT go in /health*: model parameters, hyperparameters, training data — exposing them lets attackers reconstruct your model.


### How do I add a property-based test for input invariance?

Property-based: assert that the prediction doesn't depend on dict ordering.


In [9]:
def test_predict_one_order_invariance(n=10):
    rng = np.random.default_rng(0)
    failures = 0
    for _ in range(n):
        i = rng.integers(0, len(X))
        row = X.iloc[i].to_dict()
        rev = dict(reversed(list(row.items())))
        if abs(predict_one(row)['prob_up'] - predict_one(rev)['prob_up']) > 1e-9:
            failures += 1
    assert failures == 0, f'{failures}/{n} order-dependence failures'
    print(f'order-invariance test passed ({n} rows)')

test_predict_one_order_invariance()

order-invariance test passed (10 rows)


*Why property tests over single-row tests*: a single row only catches the bug if its order *happens* to differ from training. Property tests over many random orderings catch it deterministically. *Common mistake*: tolerance too tight (e.g. 1e-15) — fails on legitimate floating-point noise. 1e-9 is comfortably above noise but well below any real bug magnitude.


---
## 4. More: async, logging, ops endpoints

Three production-ready patterns the basic deployment cheatsheet doesn't cover.


### How do I write an async FastAPI endpoint?

`async def` for any handler that does I/O (HTTP calls, DB queries). Use `await` for the I/O steps.


In [10]:
deploy_async_code = '''
from fastapi import FastAPI
import asyncio

app = FastAPI()

@app.get("/slow-data")
async def slow_data(item_id: int):
    # Simulate I/O — the event loop can serve other requests during this await.
    await asyncio.sleep(0.5)
    return {"item_id": item_id, "data": "fetched"}

@app.get("/cpu-bound")
def cpu_bound(n: int):
    # CPU-bound work — don't use async; FastAPI runs sync handlers in a threadpool.
    return {"sum": sum(i * i for i in range(n))}
'''
print(deploy_async_code)


from fastapi import FastAPI
import asyncio

app = FastAPI()

@app.get("/slow-data")
async def slow_data(item_id: int):
    # Simulate I/O — the event loop can serve other requests during this await.
    await asyncio.sleep(0.5)
    return {"item_id": item_id, "data": "fetched"}

@app.get("/cpu-bound")
def cpu_bound(n: int):
    # CPU-bound work — don't use async; FastAPI runs sync handlers in a threadpool.
    return {"sum": sum(i * i for i in range(n))}



*When to use async vs sync*: async for I/O-bound (DB, HTTP, file reads); sync (def) for CPU-bound (model inference). *Common mistake*: making everything async — async only helps when you `await` something. CPU-bound async functions block the event loop.


### How do I structure logs for production?

Use `structlog` or stdlib `logging` with JSON formatter. Always include a correlation/request ID.


In [11]:
import logging
import json
from uuid import uuid4

# Bare-bones JSON-style structured logging.
class JSONFormatter(logging.Formatter):
    def format(self, record):
        return json.dumps({
            'time':    self.formatTime(record),
            'level':   record.levelname,
            'logger':  record.name,
            'msg':     record.getMessage(),
            **getattr(record, 'extra', {}),
        })

handler = logging.StreamHandler()
handler.setFormatter(JSONFormatter())
log = logging.getLogger('app')
log.addHandler(handler); log.setLevel(logging.INFO)

# Per-request correlation ID — log it on every line.
request_id = str(uuid4())
log.info('predicting', extra={'extra': {'request_id': request_id, 'symbol': 'BTC'}})

{"time": "2026-04-28 09:06:53,858", "level": "INFO", "logger": "app", "msg": "predicting", "request_id": "e01fff68-c3ea-4349-809f-bdc6ec62b5ce", "symbol": "BTC"}


*Why structured*: makes logs queryable in any log aggregator (Datadog, Loki, CloudWatch). *Common mistake*: human-readable f-strings in logs — searchable as text; not searchable as fields. Always use `extra=` for variable data.


### What's the difference between liveness and readiness probes?

**Liveness** (`/healthz`): is the process alive? Returning 200 means "don't restart me". **Readiness** (`/ready`): am I ready to serve traffic? Failing means "send no requests".


In [12]:
ops_endpoints = '''
from fastapi import FastAPI, Response
import joblib, os

app = FastAPI()
MODEL = None

@app.on_event("startup")
async def warm_up():
    global MODEL
    MODEL = joblib.load(os.environ["MODEL_PATH"])

# Liveness: is the process alive? Always 200 if it can respond.
@app.get("/healthz")
def healthz():
    return {"status": "alive"}

# Readiness: ready to serve? Need MODEL loaded + dependencies reachable.
@app.get("/ready")
def ready(response: Response):
    if MODEL is None:
        response.status_code = 503
        return {"status": "loading", "reason": "model not warm"}
    return {"status": "ready"}
'''
print(ops_endpoints)


from fastapi import FastAPI, Response
import joblib, os

app = FastAPI()
MODEL = None

@app.on_event("startup")
async def warm_up():
    global MODEL
    MODEL = joblib.load(os.environ["MODEL_PATH"])

# Liveness: is the process alive? Always 200 if it can respond.
@app.get("/healthz")
def healthz():
    return {"status": "alive"}

# Readiness: ready to serve? Need MODEL loaded + dependencies reachable.
@app.get("/ready")
def ready(response: Response):
    if MODEL is None:
        response.status_code = 503
        return {"status": "loading", "reason": "model not warm"}
    return {"status": "ready"}



*Why two endpoints*: Kubernetes/load-balancers need both. Liveness drives restarts; readiness drives traffic routing. *Common mistake*: implementing only `/health` and using it for both — a slow startup gets killed because liveness fails before the model warms up.
